# Annotation Agent — Auto-labeling & Quality Report

| | |
|---|---|
| **Dataset** | `data/raw/unified_dataset.csv` (1 417 rows) |
| **Modality** | Text |
| **Model** | `distilbert-base-uncased-finetuned-sst-2-english` |
| **Task** | Binary sentiment classification (positive / negative) |

**Structure:**
1. Auto-labeling with DistilBERT
2. Annotation specification (`annotation_spec.md`)
3. Quality metrics (Cohen's κ vs. original labels)
4. LabelStudio export
5. **BONUS** — Human-in-the-loop (flag low-confidence)

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path("..").resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, JSON
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from agents.annotation_agent import AnnotationAgent

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

RAW = ROOT / "data" / "raw" / "unified_dataset.csv"
assert RAW.exists(), f"Run python run_agent.py first! ({RAW})"

df_raw = pd.read_csv(RAW)
# pandas-2 safe sampling per source
parts = [grp.sample(min(100, len(grp)), random_state=42)
         for _, grp in df_raw.groupby("source")]
sample = pd.concat(parts, ignore_index=True)
print(f"Working sample: {len(sample)} rows")
print(sample["source"].value_counts().to_string())
sample.head(3)


---
## 1. Auto-labeling with DistilBERT 🤖

In [ ]:
agent = AnnotationAgent(
    modality='text',
    labels=['positive', 'negative'],
    text_model='distilbert-base-uncased-finetuned-sst-2-english',
    zero_shot=False,
    batch_size=64,
    confidence_threshold=0.70,
    output_dir=ROOT / 'data' / 'annotations',
)
print('Agent created:', agent)

In [ ]:
%%time
df_labeled = agent.auto_label(sample)
print(f'\nLabeled: {len(df_labeled)} rows')
print('\nPredicted label distribution:')
print(df_labeled['predicted_label'].value_counts().to_string())
print(f'\nMean confidence: {df_labeled["confidence"].mean():.3f}')
df_labeled[['text', 'label', 'predicted_label', 'confidence']].head(6)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lp = {'positive': '#66BB6A', 'negative': '#EF5350'}

# Predicted label distribution
cnt = df_labeled['predicted_label'].value_counts()
axes[0].bar(cnt.index, cnt.values,
            color=[lp.get(l, '#90A4AE') for l in cnt.index], edgecolor='white')
axes[0].set_title('Predicted label distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(cnt.values):
    axes[0].text(i, v + 1, str(v), ha='center')

# Confidence histogram
axes[1].hist(df_labeled['confidence'], bins=30, color='#42A5F5', edgecolor='none')
axes[1].axvline(agent.confidence_threshold, color='#EF5350', lw=2, ls='--',
                label=f'threshold={agent.confidence_threshold}')
axes[1].axvline(df_labeled['confidence'].mean(), color='black', lw=1.5, ls='-',
                label=f'mean={df_labeled["confidence"].mean():.2f}')
axes[1].set_title('Confidence score distribution', fontweight='bold')
axes[1].set_xlabel('Confidence')
axes[1].legend(fontsize=9)

# Confidence by source
src_conf = df_labeled.groupby('source')['confidence'].mean().sort_values()
src_conf.plot.barh(ax=axes[2], color='#7E57C2', edgecolor='white')
axes[2].axvline(agent.confidence_threshold, color='#EF5350', lw=2, ls='--')
axes[2].set_title('Mean confidence by source', fontweight='bold')
axes[2].set_xlabel('Mean confidence')
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(ROOT / 'data/annotations/aplot_label_confidence.png', bbox_inches='tight')
plt.show()

---
## 2. Annotation Specification 📋

In [ ]:
spec_path = agent.generate_spec(
    df=sample,
    task='sentiment_classification',
    text_col='text',
    label_col='label',
    n_examples=3,
)
print(f'Spec saved to: {spec_path}')
spec_text = open(spec_path, encoding='utf-8').read()
display(Markdown(spec_text))

---
## 3. Quality Metrics 📊

We compare auto-labels against the **original dataset labels** as the reference.

> **Note on κ interpretation (Landis & Koch scale):**
> - κ < 0.20 → Slight agreement
> - 0.21–0.40 → Fair
> - 0.41–0.60 → Moderate
> - 0.61–0.80 → **Substantial** ← target for NLP annotation
> - 0.81–1.00 → Almost perfect

In [ ]:
metrics = agent.check_quality(
    df_labeled,
    reference_col='label',
    pred_col='predicted_label',
    conf_col='confidence',
)

print('=== Quality Metrics ===')
for k, v in metrics.items():
    print(f'  {k:<25}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix (auto vs. original)
valid = df_labeled[['label', 'predicted_label']].dropna()
valid = valid[valid['predicted_label'] != 'unknown']
cm = confusion_matrix(valid['label'], valid['predicted_label'],
                      labels=['negative', 'positive'])
disp = ConfusionMatrixDisplay(cm, display_labels=['negative', 'positive'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(
    f'Auto vs. Original labels\nCohen κ = {metrics["kappa"]:.3f}  '
    f'Agree = {metrics["percent_agreement"]:.1f}%',
    fontweight='bold'
)

# Confidence by correctness
valid2 = df_labeled[['label', 'predicted_label', 'confidence']].dropna()
valid2 = valid2[valid2['predicted_label'] != 'unknown'].copy()
valid2['correct'] = valid2['label'] == valid2['predicted_label']
for correct, color, label in [(True, '#66BB6A', 'Correct'), (False, '#EF5350', 'Wrong')]:
    sub = valid2[valid2['correct'] == correct]['confidence']
    axes[1].hist(sub, bins=20, alpha=0.6, color=color, label=f'{label} (n={len(sub)})')
axes[1].axvline(agent.confidence_threshold, color='black', ls='--', lw=1.5,
                label=f'threshold={agent.confidence_threshold}')
axes[1].set_title('Confidence: correct vs. wrong predictions', fontweight='bold')
axes[1].set_xlabel('Confidence')
axes[1].legend(fontsize=9)

# Cohen's κ gauge
kappa = metrics.get('kappa') or 0
gauge_colors = ['#EF5350','#FF7043','#FFA726','#66BB6A','#43A047']
thresholds = [0.20, 0.40, 0.60, 0.80, 1.0]
labels_g = ['Slight','Fair','Moderate','Substantial','Perfect']
wedge_sizes = [20, 20, 20, 20, 20]
axes[2].pie(wedge_sizes, colors=gauge_colors, startangle=180,
            counterclock=False, wedgeprops={'width': 0.4})
# Mark κ position
angle = 180 - (kappa * 180)
import math
x = 0.65 * math.cos(math.radians(angle))
y = 0.65 * math.sin(math.radians(angle))
axes[2].annotate('', xy=(x, y), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='black', lw=2.5))
axes[2].text(0, -0.15, f'κ = {kappa:.3f}', ha='center', fontsize=14, fontweight='bold')
for i, (t, lbl) in enumerate(zip(thresholds, labels_g)):
    a = 180 - (t - 0.10) * 180
    axes[2].text(0.85 * math.cos(math.radians(a)),
                 0.85 * math.sin(math.radians(a)),
                 lbl, ha='center', fontsize=7, color='white', fontweight='bold')
axes[2].set_title("Cohen's κ gauge", fontweight='bold')
axes[2].set_ylim(-0.5, 1.1)

plt.tight_layout()
plt.savefig(ROOT / 'data/annotations/aplot_quality.png', bbox_inches='tight')
plt.show()

### Quality breakdown per source

In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score

rows = []
for src in df_labeled['source'].unique():
    sub = df_labeled[df_labeled['source'] == src][['label', 'predicted_label', 'confidence']].dropna()
    sub = sub[sub['predicted_label'] != 'unknown']
    if len(sub) < 2:
        continue
    try:
        kappa = cohen_kappa_score(sub['label'], sub['predicted_label'])
    except Exception:
        kappa = None
    rows.append({
        'source': src,
        'n': len(sub),
        'kappa': round(kappa, 3) if kappa is not None else None,
        'accuracy': round(accuracy_score(sub['label'], sub['predicted_label']), 3),
        'mean_conf': round(sub['confidence'].mean(), 3),
    })

per_src_df = pd.DataFrame(rows)
display(per_src_df.style.background_gradient(subset=['kappa','accuracy','mean_conf'],
                                               cmap='RdYlGn', vmin=0, vmax=1).format(precision=3))

---
## 4. LabelStudio Export 📤

In [ ]:
export_result = agent.export_to_labelstudio(
    df_labeled,
    text_col='text',
    pred_col='predicted_label',
    conf_col='confidence',
    task_name='sentiment',
)

print('=== LabelStudio Export ===')
for k, v in export_result.items():
    if k != 'tasks':
        print(f'  {k}: {v}')

# Preview first task
print('\nFirst task (preview):')
import json
print(json.dumps(export_result['tasks'][0], indent=2, ensure_ascii=False))

In [ ]:
# Show the label config XML for pasting into LabelStudio UI
config_path = export_result['config_path']
print('=== LabelStudio Config XML ===')
print('(Copy this into Project → Settings → Labeling Interface)\n')
print(open(config_path).read())

---
## 5. Bonus — Human-in-the-Loop (HITL) 🚦

The agent flags all predictions with `confidence < 0.70` for manual review.
These rows are saved to `data/annotations/hitl_review.csv` — ready to be
imported into LabelStudio as a priority annotation queue.

High-confidence predictions are auto-accepted and saved to `hitl_auto_accepted.csv`.

In [ ]:
low_conf_df = agent.flag_low_confidence(df_labeled, threshold=0.70)
high_conf_df = df_labeled[df_labeled['confidence'] >= 0.70]

print(f'Total rows       : {len(df_labeled)}')
print(f'Auto-accepted    : {len(high_conf_df)}  (conf ≥ 0.70)')
print(f'Needs human review: {len(low_conf_df)}  (conf < 0.70)')
print(f'\nReview rate: {len(low_conf_df)/len(df_labeled)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
thr = agent.confidence_threshold

# HITL split pie
n_low = len(low_conf_df)
n_high = len(high_conf_df)
axes[0].pie([n_high, n_low],
            labels=[f'Auto-accepted\n({n_high})', f'Needs review\n({n_low})'],
            colors=['#66BB6A', '#EF5350'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[0].set_title('HITL split', fontweight='bold')

# Confidence distribution colored by HITL decision
axes[1].hist(high_conf_df['confidence'], bins=25, color='#66BB6A',
             alpha=0.75, label=f'Auto-accepted (n={n_high})')
axes[1].hist(low_conf_df['confidence'], bins=25, color='#EF5350',
             alpha=0.75, label=f'Needs review (n={n_low})')
axes[1].axvline(thr, color='black', ls='--', lw=2, label=f'threshold={thr}')
axes[1].set_title('Confidence split at HITL threshold', fontweight='bold')
axes[1].set_xlabel('Confidence')
axes[1].legend(fontsize=9)

# Low-confidence examples by source
if len(low_conf_df) > 0 and 'source' in low_conf_df.columns:
    lc_src = low_conf_df['source'].value_counts()
    axes[2].bar(lc_src.index, lc_src.values, color='#EF5350', edgecolor='white')
    axes[2].set_title('Low-confidence rows by source', fontweight='bold')
    axes[2].set_ylabel('Count')
    axes[2].tick_params(axis='x', rotation=20)
else:
    axes[2].text(0.5, 0.5, 'No low-confidence rows', ha='center', va='center',
                 transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(ROOT / 'data/annotations/aplot_hitl.png', bbox_inches='tight')
plt.show()

print('\nSample low-confidence rows:')
display(low_conf_df[['text', 'label', 'predicted_label', 'confidence']]
        .sort_values('confidence').head(8).reset_index(drop=True))

### Export low-confidence subset to LabelStudio for priority review

In [ ]:
if len(low_conf_df) > 0:
    hitl_export = agent.export_to_labelstudio(
        low_conf_df,
        output_filename='labelstudio_hitl_review.json',
    )
    print(f"HITL tasks exported: {hitl_export['n_tasks']} → {hitl_export['json_path']}")
else:
    print('No low-confidence examples to export.')

---
## Summary

In [ ]:
print('=' * 55)
print('  ANNOTATION AGENT — FINAL SUMMARY')
print('=' * 55)
print(f'  Backend           : {agent._backend}')
print(f'  Rows labeled      : {len(df_labeled)}')
print(f'  Mean confidence   : {df_labeled["confidence"].mean():.3f}')
print(f"  Cohen's κ         : {metrics['kappa']:.3f}")
print(f"  Agreement         : {metrics['percent_agreement']:.1f}%")
print(f'  Low-conf (HITL)   : {len(low_conf_df)} ({len(low_conf_df)/len(df_labeled)*100:.1f}%)')
print('\n  Output files:')
import os
ann_dir = ROOT / 'data' / 'annotations'
for f in sorted(ann_dir.iterdir()):
    print(f'    {f.name}  ({f.stat().st_size // 1024} KB)')
print('=' * 55)